# Vocabulary & Data Preparation

Convert text → numerical form

Add special tokens (<sos>, <eos>)

Prepare tensors for model input

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -----------------------------
# Vocabulary Setup
# -----------------------------
vocab = ["<pad>", "<sos>", "<eos>", "i", "like", "ai"]

# Word ↔ Index mapping
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# Encode sentence → indices
def encode(sentence):
    return [word2idx[w] for w in sentence]

# Decode indices → sentence
def decode(indices):
    return [idx2word[i] for i in indices]

# Example input-output pair
input_sentence = ["i", "like", "ai"]
target_sentence = ["ai", "like", "i"]

# Convert to tensors
input_tensor = torch.tensor([encode(input_sentence)])

# Add <sos> (start) and <eos> (end)
target_tensor = torch.tensor([
    [word2idx["<sos>"]] + encode(target_sentence) + [word2idx["<eos>"]]
])

# Encoder (LSTM)

Reads input sequence

Produces:

hidden state (h) → short-term memory

cell state (c) → long-term memory


In [ ]:
# -----------------------------
# Encoder (LSTM)
# -----------------------------
class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        # Token → embedding vector
        self.embedding = nn.Embedding(vocab_size, embed_size)

        # LSTM layer
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

    def forward(self, x):
        # Convert tokens → embeddings
        embedded = self.embedding(x)

        # Pass through LSTM
        outputs, (hidden, cell) = self.lstm(embedded)

        # Return both hidden and cell states
        return hidden, cell

# Decoder (LSTM)

Generates output one token at a time

Uses:

Previous hidden state

Previous cell state

In [ ]:
# -----------------------------
# Decoder (LSTM)
# -----------------------------
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_size)

        # LSTM takes previous states as input
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)

        # Convert hidden state → vocabulary scores
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        # Step 1: Embed input token
        embedded = self.embedding(x)

        # Step 2: LSTM forward pass with previous states
        outputs, (hidden, cell) = self.lstm(embedded, (hidden, cell))

        # Step 3: Predict next token
        predictions = self.fc(outputs)

        return predictions, hidden, cell

# Seq2Seq Model (Encoder + Decoder)

Connect encoder and decoder

Implements teacher forcing

In [ ]:
# -----------------------------
# Seq2Seq Model
# -----------------------------
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, trg):
        # Encode input → get hidden & cell states
        hidden, cell = self.encoder(src)

        outputs = []

        # First input to decoder = <sos>
        input_token = trg[:, 0].unsqueeze(1)

        # Generate sequence step-by-step
        for t in range(1, trg.size(1)):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs.append(output)

            # Teacher forcing: use actual target token
            input_token = trg[:, t].unsqueeze(1)

        return torch.cat(outputs, dim=1)

# Model Initialization & Training Setup

Define architecture size

Set loss and optimizer

In [ ]:
# -----------------------------
# Training Setup
# -----------------------------
vocab_size = len(vocab)
embed_size = 8
hidden_size = 16

encoder = Encoder(vocab_size, embed_size, hidden_size)
decoder = Decoder(vocab_size, embed_size, hidden_size)

model = Seq2Seq(encoder, decoder)

# Loss: compares predicted tokens vs actual tokens
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Training Loop

Learn model parameters via backpropagation

In [ ]:
# -----------------------------
# Training Loop
# -----------------------------
for epoch in range(300):
    optimizer.zero_grad()

    # Forward pass
    output = model(input_tensor, target_tensor)

    # Reshape for loss computation
    output = output.view(-1, vocab_size)

    # Ignore <sos> in target
    target = target_tensor[:, 1:].reshape(-1)

    # Compute loss
    loss = criterion(output, target)

    # Backpropagation
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 1.8440
Epoch 50, Loss: 0.0255
Epoch 100, Loss: 0.0072
Epoch 150, Loss: 0.0042
Epoch 200, Loss: 0.0028
Epoch 250, Loss: 0.0021


# Inference (Prediction Phase)

Generate output without teacher forcing

Uses model’s own predictions

In [ ]:
# -----------------------------
# Inference
# -----------------------------
def predict(model, src, max_len=10):
    model.eval()

    with torch.no_grad():
        # Encode input
        hidden, cell = model.encoder(src)

        # Start with <sos>
        input_token = torch.tensor([[word2idx["<sos>"]]])
        result = []

        # Generate tokens step-by-step
        for _ in range(max_len):
            output, hidden, cell = model.decoder(input_token, hidden, cell)

            pred = output.argmax(2).item()

            # Stop at <eos>
            if pred == word2idx["<eos>"]:
                break

            result.append(pred)

            # Feed prediction back
            input_token = torch.tensor([[pred]])

    return decode(result)

print("Prediction:", predict(model, input_tensor))

Prediction: ['ai', 'like', 'i']
